# Factuality Audit After Rewriting

This notebook audits whether rewrites generated in `llm_simulation_workbench.ipynb` remain consistent with the original text.

Instead of training fake vs real, it measures contradiction between original and rewritten texts (risk of becoming potentially false after rewriting).

## 1) Imports

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

## 2) Configuracao

Ajuste caminhos, selecao de datasets e limiares de auditoria. Por padrao, a leitura vem de `../output` (gerado no notebook de simulacao LLM).

In [ ]:
# Pasta com CSVs reescritos exportados no notebook de reescrita
run_id_env = os.getenv("RUN_ID")
if run_id_env:
    RUN_ID = run_id_env
    INPUT_DIR = Path("../output/runs") / RUN_ID / "rewritten"
    AUDIT_OUTPUT_DIR = Path("../output/runs") / RUN_ID / "audit" / "LocalAudit"
else:
    RUN_ID = None
    INPUT_DIR = Path("../output/rewritten")
    AUDIT_OUTPUT_DIR = Path("../output/audit/LocalAudit")

# Selecao global do dataset:
# - "ALL" para processar todos os CSVs da pasta INPUT_DIR
# - nome de arquivo (ex.: "science_rewritten_df.csv") para processar apenas um
DATASET_SELECTOR = "ALL"

# Colunas dos textos
ORIGINAL_COLUMN = "description"
REWRITTEN_COLUMN = "rewritten_news"
ROW_ID_COLUMN = None  # opcional, ex.: "id"

# Modelo NLI para detectar contradicao
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# Limites para marcar potencialmente falsa apos reescrita
CONTRADICTION_THRESHOLD = 0.60
DELTA_THRESHOLD = 0.20  # contradicao - entailment

MAX_ROWS_PER_DATASET = 1000

# Pasta de saida das auditorias
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuracao carregada.")
if RUN_ID:
    print(f"RUN_ID: {RUN_ID}")
else:
    print("RUN_ID nao definido. Usando pastas padrao em ../output.")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"DATASET_SELECTOR: {DATASET_SELECTOR}")
print(f"AUDIT_OUTPUT_DIR: {AUDIT_OUTPUT_DIR}")

## 3) Funcoes utilitarias

In [ ]:
from utils.bert_audit_functions import (
    consistency_flag,
    nli_pair_scores,
    read_dataset,
    validate_pair_columns,
)
from utils.run_report import append_execution_report, resolve_execution_report_path

## 4) Carregar dados pareados

In [ ]:
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Pasta de entrada nao encontrada: {INPUT_DIR}")

if DATASET_SELECTOR.strip().upper() == "ALL":
    selected_files = sorted(INPUT_DIR.glob("*.csv"), key=lambda p: p.name.lower())
else:
    selected_file = INPUT_DIR / DATASET_SELECTOR
    if not selected_file.exists():
        raise FileNotFoundError(f"Arquivo selecionado nao encontrado: {selected_file}")
    selected_files = [selected_file]

if not selected_files:
    raise FileNotFoundError(f"Nenhum CSV encontrado em: {INPUT_DIR}")

dataset_payloads = []

for file_path in selected_files:
    df = read_dataset(file_path).copy()
    validate_pair_columns(df, ORIGINAL_COLUMN, REWRITTEN_COLUMN)

    df[ORIGINAL_COLUMN] = df[ORIGINAL_COLUMN].fillna("").astype(str).str.strip()
    df[REWRITTEN_COLUMN] = df[REWRITTEN_COLUMN].fillna("").astype(str).str.strip()
    df = df[(df[ORIGINAL_COLUMN] != "") & (df[REWRITTEN_COLUMN] != "")].copy()
    df = df.head(MAX_ROWS_PER_DATASET).copy()

    dataset_payloads.append({
        "file_path": file_path,
        "file_name": file_path.name,
        "dataset_name": file_path.stem,
        "paired_df": df,
    })

print(f"Datasets selecionados: {len(dataset_payloads)}")
for payload in dataset_payloads:
    print(f"- {payload['file_name']}: {len(payload['paired_df'])} pares")

dataset_payloads[0]["paired_df"][[ORIGINAL_COLUMN, REWRITTEN_COLUMN]].head()

## 5) Carregar modelo NLI e avaliar contradicao

In [ ]:
audit_started_at = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
model.eval()

id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}

all_audits = []
summary_rows = []

for payload in dataset_payloads:
    paired_df = payload["paired_df"]
    rows = []

    for idx, row in paired_df.iterrows():
        original_text = row[ORIGINAL_COLUMN]
        rewritten_text = row[REWRITTEN_COLUMN]

        scores = nli_pair_scores(tokenizer, model, id2label, original_text, rewritten_text)
        flag = consistency_flag(
            scores["entailment"],
            scores["contradiction"],
            CONTRADICTION_THRESHOLD,
            DELTA_THRESHOLD,
        )

        out = {
            "row_index": int(idx),
            "entailment": scores["entailment"],
            "contradiction": scores["contradiction"],
            "neutral": scores["neutral"],
            "consistency_flag": flag,
        }

        if ROW_ID_COLUMN and ROW_ID_COLUMN in paired_df.columns:
            out["row_id"] = row[ROW_ID_COLUMN]

        rows.append(out)

    result_df = pd.DataFrame(rows)
    audit_df = paired_df.join(result_df.set_index("row_index"), how="left")
    audit_df["source_file"] = payload["file_name"]
    audit_df["dataset_name"] = payload["dataset_name"]

    all_audits.append(audit_df)

    suspects_count = int((audit_df["consistency_flag"] == "potencialmente_falsa_apos_reescrita").sum())
    summary_rows.append(
        {
            "dataset_name": payload["dataset_name"],
            "source_file": payload["file_name"],
            "rows": int(len(audit_df)),
            "suspects": suspects_count,
            "suspect_rate": (suspects_count / len(audit_df)) if len(audit_df) > 0 else 0.0,
        }
    )

audit_all_df = pd.concat(all_audits, ignore_index=True) if all_audits else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows).sort_values("suspect_rate", ascending=False)

print("Resumo da auditoria por dataset:")
display(summary_df)

if not audit_all_df.empty:
    print("Resumo geral de flags:")
    print(audit_all_df["consistency_flag"].value_counts(dropna=False))

audit_elapsed_seconds = time.perf_counter() - audit_started_at

audit_all_df[[
    "dataset_name",
    ORIGINAL_COLUMN,
    REWRITTEN_COLUMN,
    "entailment",
    "contradiction",
    "consistency_flag",
]].head(10)

## 6) Salvar e inspecionar casos mais criticos

In [ ]:
if audit_all_df.empty:
    raise ValueError("Nenhum resultado de auditoria foi gerado.")

per_file_outputs = []
for dataset_name, dataset_audit in audit_all_df.groupby("dataset_name", dropna=False):
    output_file = AUDIT_OUTPUT_DIR / f"{dataset_name}_consistency_audit.csv"
    dataset_audit.to_csv(output_file, index=False)
    per_file_outputs.append(output_file)

combined_output = AUDIT_OUTPUT_DIR / "all_datasets_consistency_audit.csv"
audit_all_df.to_csv(combined_output, index=False)

summary_output = AUDIT_OUTPUT_DIR / "audit_summary.csv"
summary_df.to_csv(summary_output, index=False)

print("Arquivos de auditoria salvos:")
for output_file in per_file_outputs:
    print(f"- {output_file}")
print(f"- {combined_output}")
print(f"- {summary_output}")

run_id_for_report, report_path = resolve_execution_report_path()

append_execution_report(
    report_path=report_path,
    notebook_name="bert_fake_real_workbench.ipynb",
    section_title="Local consistency audit execution",
    run_id=run_id_for_report,
    details={
        "rows_audited": int(len(audit_all_df)),
        "audit_execution_seconds": round(audit_elapsed_seconds, 3),
        "pretrained_model_used": NLI_MODEL,
        "local_trained_model_used": NLI_MODEL,
        "local_model_data_rows_used": int(sum(len(payload["paired_df"]) for payload in dataset_payloads)),
        "local_model_query_details": {
            "dataset_selector": DATASET_SELECTOR,
            "input_dir": str(INPUT_DIR),
            "original_column": ORIGINAL_COLUMN,
            "rewritten_column": REWRITTEN_COLUMN,
        },
    },
)

print(f"Relatorio de execucao atualizado: {report_path}")

suspects = audit_all_df.sort_values("contradiction", ascending=False).head(20)
suspects[[
    "dataset_name",
    ORIGINAL_COLUMN,
    REWRITTEN_COLUMN,
    "entailment",
    "contradiction",
    "consistency_flag",
]]